In [4]:
import json
import pandas as pd
import io # 虽然在这个简单场景下不是必须的，但处理二进制流时引入是个好习惯

def convert_jsonl_to_parquet_with_images(jsonl_file_path, image_path_json_file, parquet_output_path):
    """
    将 JSONL 文件和图像路径 JSON 文件转换为 Parquet 文件，
    并将真实的图片二进制数据直接嵌入 Parquet 文件中。

    参数:
    jsonl_file_path (str): 输入的 JSONL 文件的路径。
    image_path_json_file (str): 包含 image_id 到图像路径映射的 JSON 文件的路径。
    parquet_output_path (str): 输出的 Parquet 文件的路径。
    """
    # 1. 读取 JSONL 文件
    data = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))

    # 2. 读取图像路径的 JSON 文件
    with open(image_path_json_file, 'r', encoding='utf-8') as f:
        image_paths = json.load(f)

    # 3. 准备要写入 Parquet 文件的数据
    processed_data = []
    print("开始处理数据并读取图片...")
    for i, item in enumerate(data):
        image_id = item.get("image_id")
        image_path = image_paths.get(image_id)

        if image_path:
            try:
                # --- 这是核心修改 ---
                # 读取图片文件的二进制内容
                with open(image_path, 'rb') as image_file:
                    image_bytes = image_file.read()
                # -------------------

                processed_data.append({
                    "id": image_id,
                    "question": item.get("question"),
                    # 将图片路径替换为图片的二进制数据
                    "image": image_bytes,
                    "answer": item.get("ground_truth")
                })
            except FileNotFoundError:
                print(f"警告：文件未找到，跳过 image_id '{image_id}'，路径: {image_path}")
            except Exception as e:
                print(f"警告：读取图片 '{image_path}' 时发生错误: {e}")
        else:
            print(f"警告：找不到 image_id '{image_id}' 对应的图片路径。")
        
        # 增加一个简单的进度提示
        if (i + 1) % 100 == 0:
            print(f"已处理 {i + 1} / {len(data)} 条数据...")

    print("数据处理完成，开始创建 DataFrame...")
    # 4. 创建 Pandas DataFrame
    df = pd.DataFrame(processed_data)

    # 5. 将 DataFrame 保存为 Parquet 文件
    print(f"正在将 DataFrame 保存到 Parquet 文件: {parquet_output_path}")
    try:
        # 使用 pyarrow 引擎，它能很好地处理二进制数据
        df.to_parquet(parquet_output_path, engine='pyarrow', index=False)
        print(f"成功将数据（包含图片）转换为 Parquet 文件，并保存至: {parquet_output_path}")
    except Exception as e:
        print(f"写入 Parquet 文件时出错: {e}")

# --- 使用示例 ---
# 请将下面的文件名替换为您自己的文件名
jsonl_file = "/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/validation_962.jsonl"      # 您的 JSONL 文件名
image_paths_json = "/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/web_datasets/GUIChat-processed/images/image_id2path.json" # 您的图像路径 JSON 文件名
output_parquet_file = "/mnt/petrelfs/sunhaoyu/visual-code/datasets/web_guichat/validation.parquet"  # 建议使用新文件名以作区分

# 运行转换函数
convert_jsonl_to_parquet_with_images(jsonl_file, image_paths_json, output_parquet_file)

开始处理数据并读取图片...
已处理 100 / 962 条数据...
已处理 200 / 962 条数据...
已处理 300 / 962 条数据...
已处理 400 / 962 条数据...
已处理 500 / 962 条数据...
已处理 600 / 962 条数据...
已处理 700 / 962 条数据...
已处理 800 / 962 条数据...
已处理 900 / 962 条数据...
数据处理完成，开始创建 DataFrame...
正在将 DataFrame 保存到 Parquet 文件: /mnt/petrelfs/sunhaoyu/visual-code/datasets/web_guichat/validation.parquet
成功将数据（包含图片）转换为 Parquet 文件，并保存至: /mnt/petrelfs/sunhaoyu/visual-code/datasets/web_guichat/validation.parquet
